# LangChain DeepAgents — Introduction

**DeepAgents** is a ready-to-use agent framework built on top of LangGraph.  
The idea is simple: you give it a model, your tools, and a goal — it handles the rest.

This notebook covers four introductory examples:

1. **Hello World** — the simplest possible agent
2. **Agent with tools** — giving the agent abilities it didn't have before
3. **Multi-step task** — letting the agent plan and work through a longer problem
4. **Two agents talking** — two agents having a back-and-forth conversation

## Setup

In [1]:
# Install DeepAgents if not already installed
# !pip install deepagents langchain-openai python-dotenv azure-identity

In [1]:
import warnings
warnings.filterwarnings('ignore', category=PendingDeprecationWarning)

# from langchain_core.globals import set_llm_cache
# from langchain_sqlite import SQLiteChatMessageHistory

# cache_path = "cache/langchain.db"
# set_llm_cache(SQLiteChatMessageHistory(session_id="main", connection_string=f"sqlite:///{cache_path}"))

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
)

---
## Example 1 — Hello World

The simplest deep agent: no tools, just a model and a system prompt.  
Even without custom tools, the agent already has built-in planning and file management capabilities.

In [3]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant.",
)

result = agent.invoke({"messages": "What is a deep agent and how is it different from a regular chatbot?"})
print(result["messages"][-1].content)

A deep agent is an advanced type of AI assistant designed to handle complex, multi-step tasks and provide more sophisticated interactions compared to a regular chatbot. Here are some key differences:

1. **Complex Task Management**: Deep agents can manage and execute complex tasks that require multiple steps, planning, and coordination. They can break down tasks into smaller components, track progress, and adjust plans as needed.

2. **Tool Integration**: Deep agents often have access to a variety of tools and can perform actions such as reading and writing files, searching for information, and executing code. This allows them to perform tasks beyond simple conversation.

3. **Contextual Understanding**: They maintain a deeper understanding of context, allowing them to handle more nuanced and context-dependent interactions. This includes remembering past interactions and using that information to inform current tasks.

4. **Parallel Processing**: Deep agents can handle multiple tasks s

---
## Example 2 — Agent with Tools

Tools are Python functions decorated with `@tool`.  
The agent decides on its own when and how to call them based on the user's request.

In [4]:
from langchain_core.tools import tool
from deepagents import create_deep_agent

# Define two simple tools
@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city."""
    # Simulated data — replace with a real weather API in production
    weather = {
        "Milan":  "22°C, sunny",
        "London": "14°C, cloudy",
        "Tokyo":  "28°C, humid",
        "New York": "19°C, partly cloudy",
    }
    return weather.get(city, f"No weather data available for {city}.")


@tool
def convert_celsius_to_fahrenheit(celsius: float) -> str:
    """Convert a temperature from Celsius to Fahrenheit."""
    fahrenheit = (celsius * 9 / 5) + 32
    return f"{celsius}°C = {fahrenheit:.1f}°F"


# Create the agent with the tools
agent = create_deep_agent(
    model=llm,
    tools=[get_weather, convert_celsius_to_fahrenheit],
    system_prompt="You are a weather assistant. Use the available tools to answer questions.",
)

result = agent.invoke({
    "messages": "What is the weather in Milan and Tokyo? Also give me both temperatures in Fahrenheit."
})

print(result["messages"][-1].content)

The current weather in Milan is 22°C and sunny, which is equivalent to 71.6°F. In Tokyo, the weather is 28°C and humid, which translates to 82.4°F.


Notice that the agent called the tools in the right order without being told to.  
It first fetched the weather for both cities, then converted each temperature.

---
## Example 3 — Multi-Step Task

When a task has multiple steps, DeepAgents automatically creates a plan with `write_todos` before starting.  
Here we ask the agent to analyse a small dataset step by step.

In [5]:
from langchain_core.tools import tool
from deepagents import create_deep_agent

# Simulated sales dataset
SALES_DATA = {
    "January":  12500,
    "February": 10800,
    "March":    15200,
    "April":    13900,
    "May":      17400,
    "June":     16100,
}

@tool
def get_monthly_sales(month: str) -> str:
    """Return the sales figure for a given month."""
    value = SALES_DATA.get(month.capitalize())
    if value is None:
        return f"No data for month: {month}"
    return f"{month.capitalize()} sales: ${value:,}"


@tool
def get_all_months() -> str:
    """Return all months available in the dataset."""
    return ", ".join(SALES_DATA.keys())


agent = create_deep_agent(
    model=llm,
    tools=[get_monthly_sales, get_all_months],
    system_prompt="""You are a sales analyst. 
    When asked to analyse data, first get the list of available months, 
    then retrieve each month's data, and finally summarise your findings.""",
)

result = agent.invoke({
    "messages": """Analyse our sales data for the first half of the year. 
    Identify the best month, the worst month, and the overall trend."""
})

print(result["messages"][-1].content)

Here's the analysis of the sales data for the first half of the year:

- **Best Month**: May, with sales of $17,400.
- **Worst Month**: February, with sales of $10,800.
- **Overall Trend**: The sales show an upward trend from February to May, with a slight dip in June. Starting from $10,800 in February, sales increased steadily to $17,400 in May, before slightly decreasing to $16,100 in June.


In [6]:
# You can also inspect the full conversation to see each step the agent took
for msg in result["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    if hasattr(msg, "content") and msg.content:
        print(f"[{role}]")
        print(msg.content[:300])  # first 300 chars to keep output short
        print("-" * 50)

[Human]
Analyse our sales data for the first half of the year. 
    Identify the best month, the worst month, and the overall trend.
--------------------------------------------------
[Tool]
January, February, March, April, May, June
--------------------------------------------------
[Tool]
January sales: $12,500
--------------------------------------------------
[Tool]
February sales: $10,800
--------------------------------------------------
[Tool]
March sales: $15,200
--------------------------------------------------
[Tool]
April sales: $13,900
--------------------------------------------------
[Tool]
May sales: $17,400
--------------------------------------------------
[Tool]
June sales: $16,100
--------------------------------------------------
[AI]
Here's the analysis of the sales data for the first half of the year:

- **Best Month**: May, with sales of $17,400.
- **Worst Month**: February, with sales of $10,800.
- **Overall Trend**: The sales show an upward trend from February

---
## Example 4 — Two Agents Talking to Each Other

You can wire up two independent agents and let them exchange messages in a loop.  
Each agent has its own role and responds to what the other just said.

Here we create a **Mentor** and a **Student**. The student asks questions about Python,  
the mentor answers, and the conversation continues for several turns — just like a real dialogue.

In [7]:
from deepagents import create_deep_agent

# --- Agent 1: the Mentor ---
mentor = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="""You are an experienced Python mentor talking to a junior student.
    Give clear, encouraging answers. Keep each reply to 2-3 sentences.
    End every message with a follow-up question to keep the conversation going.""",
)

# --- Agent 2: the Student ---
student = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="""You are a junior developer learning Python for data science.
    Ask genuine questions based on what the mentor just told you.
    Keep each message to 1-2 sentences — one question at a time.""",
)

# --- Conversation loop ---
NUM_TURNS = 4  # each agent speaks NUM_TURNS times

# The student opens the conversation
opening = "Hi! I just started learning Python for data science. Where should I begin?"

print("=" * 60)
print(f"[Student]\n{opening}\n")

current_message = opening

for turn in range(NUM_TURNS):
    # Mentor replies to the student
    result = mentor.invoke({"messages": current_message})
    mentor_reply = result["messages"][-1].content
    print(f"[Mentor — turn {turn + 1}]\n{mentor_reply}\n")

    # Student replies to the mentor
    result = student.invoke({"messages": mentor_reply})
    student_reply = result["messages"][-1].content
    print(f"[Student — turn {turn + 1}]\n{student_reply}\n")

    # Pass the student's reply to the mentor in the next round
    current_message = student_reply

print("=" * 60)

[Student]
Hi! I just started learning Python for data science. Where should I begin?

[Mentor — turn 1]
Welcome to the world of Python and data science! A great starting point is to familiarize yourself with Python basics, such as variables, data types, loops, and functions. Once you're comfortable with these, you can move on to libraries like NumPy and pandas, which are essential for data manipulation and analysis.

Have you already installed Python and set up your development environment?

[Student — turn 1]
Not yet. Could you guide me on how to install Python and set up a development environment for data science?

[Mentor — turn 2]
To set up a Python development environment for data science, follow these steps:

1. **Install Python**: Download and install the latest version of Python from the [official website](https://www.python.org/downloads/). Ensure you check the option to add Python to your PATH during installation.

2. **Install Anaconda**: Anaconda is a popular distribution f

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-GC5hssZM601ksKMcM8WND7Hu on tokens per min (TPM): Limit 30000, Used 29394, Requested 1924. Please try again in 2.636s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

---
## What's next?

Once you're comfortable with these basics, the next capabilities to explore are:

| Feature | What it does |
|---|---|
| **Filesystem** | The agent can read and write files in a workspace |
| **Memory** (`AGENTS.md`) | Preferences and context that persist across sessions |
| **Skills** (`SKILL.md`) | Specialised instructions loaded only when needed |
| **Subagents** | Delegate subtasks to specialist child agents |
| **Streaming** | Get responses token by token as the agent works |

See `deepagents_langchain.md` in this repo for a full reference.